# SDE-style PSD diffuse-only notebook with transformer-only conditioning

This notebook trains a diffuse-only `x`-parameterized diffusion model on PSD data using your `SDEBackbone` idea:

- target is always the diffuse RGB image
- the U-Net sees only the noisy diffuse sample `x_t`
- conditioning is injected only in the transformer bottleneck
- conditioning can be either raw glossy RGB or a 1-channel edge map
- periodic eval uses a restoration-style reverse path from a noisy glossy prior

Use `CONDITION_MODE = 'glossy_rgb'` for 3-channel glossy conditioning or `CONDITION_MODE = 'edge_1ch'` for a 1-channel structural condition.


In [ ]:
from __future__ import annotations

import math
import random
import re
import sys
from collections import OrderedDict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / 'Run_Training.py').exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / 'Run_Training.py').exists() and (candidate / 'SDEBackbone.py').exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ParamDiffuser as diff
import SDEBackbone
import Transformer


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = ROOT / 'models' / '32'
CHECKPOINTS_DIR = ROOT / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo root: {ROOT}')
print(f'device: {DEVICE}')


In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 10
NOISE_STEPS = 200
EPOCHS = 2000
LR = 1e-4
FINAL_LR = 1e-5
WARMUP_EPOCHS = 100
EMA_DECAY = 0.9999
DEPTH = 4

TRAIN_TYPE = 'x'
CONDITION_MODE = 'edge_1ch'  # switch between 'edge_1ch' and 'glossy_rgb'
LOSS_KIND = 'l1'  # 'l1' or 'mse'
GLOBAL_LOSS_WEIGHT = 1.0
WEIGHTED_LOSS_WEIGHT = 1.0
HIGHLIGHT_WEIGHT = 4.0
HIGHLIGHT_QUANTILE = 0.98

EVAL_EVERY = 200
CHECKPOINT_EVERY = 500
VAL_BATCHES = 4
VAL_CASE_INDEX = 0
EVAL_SEED = 123
RESTORE_START_STEP = 40
RESTORE_DETERMINISTIC = True
RESTORE_CLIP_X0 = True

PSD_ROOT = Path('/Users/27171653/Desktop/PhD/Highlight-modelling/PSD_Dataset/PSD_Dataset')
TRAIN_GLOSSY_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_specular'
TRAIN_DIFFUSE_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_diffuse'
VAL_GLOSSY_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_specular'
VAL_DIFFUSE_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_diffuse'

if TRAIN_TYPE != 'x':
    raise ValueError(f'This notebook is fixed to x-parameterization, got {TRAIN_TYPE!r}')
if CONDITION_MODE not in {'edge_1ch', 'glossy_rgb'}:
    raise ValueError(f"CONDITION_MODE must be 'edge_1ch' or 'glossy_rgb', got {CONDITION_MODE!r}")
if LOSS_KIND not in {'l1', 'mse'}:
    raise ValueError(f"LOSS_KIND must be 'l1' or 'mse', got {LOSS_KIND!r}")
if not 0 <= RESTORE_START_STEP < NOISE_STEPS:
    raise ValueError(f'RESTORE_START_STEP must be in [0, {NOISE_STEPS - 1}], got {RESTORE_START_STEP}')

CONDITION_CHANNELS = 1 if CONDITION_MODE == 'edge_1ch' else 3

SAVE_STEM = f'SDE_DiffuseOnly_{CONDITION_MODE}_Flex{IMAGE_SIZE}'
TRAINER_SAVE_PATH = CHECKPOINTS_DIR / f'{SAVE_STEM}_checkpoint'
FINAL_MODEL_PATH = MODELS_DIR / f'{SAVE_STEM}.pth'

print(f'PSD root: {PSD_ROOT}')
print(f'condition mode: {CONDITION_MODE}')
print(f'conditioning channels: {CONDITION_CHANNELS}')
print(f'restore start step: {RESTORE_START_STEP}')
print(f'final model path: {FINAL_MODEL_PATH}')


In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
PIL_BILINEAR = getattr(Image, 'Resampling', Image).BILINEAR


def normalize_image_key(name: str) -> str:
    stem = Path(name).stem.lower()
    stem = stem.replace('specular', '').replace('glossy', '').replace('diffuse', '')
    return re.sub(r'[^a-z0-9]+', '', stem)


def pair_image_paths(glossy_dir: Path, diffuse_dir: Path):
    glossy_candidates = [path for path in glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS]
    diffuse_candidates = [path for path in diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS]

    glossy_files = {path.name: path for path in glossy_candidates}
    diffuse_files = {path.name: path for path in diffuse_candidates}
    exact_names = sorted(set(glossy_files) & set(diffuse_files))
    if exact_names:
        return [(glossy_files[name], diffuse_files[name], name) for name in exact_names]

    glossy_by_key = {}
    for path in glossy_candidates:
        key = normalize_image_key(path.name)
        if key in glossy_by_key:
            raise RuntimeError(f'Duplicate glossy normalized key {key!r} in {glossy_dir}')
        glossy_by_key[key] = path

    diffuse_by_key = {}
    for path in diffuse_candidates:
        key = normalize_image_key(path.name)
        if key in diffuse_by_key:
            raise RuntimeError(f'Duplicate diffuse normalized key {key!r} in {diffuse_dir}')
        diffuse_by_key[key] = path

    common_keys = sorted(set(glossy_by_key) & set(diffuse_by_key))
    if not common_keys:
        raise RuntimeError(f'No paired PSD samples found in {glossy_dir} and {diffuse_dir}')

    return [(glossy_by_key[key], diffuse_by_key[key], glossy_by_key[key].name) for key in common_keys]


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert('RGB')
    image = image.resize((image_size, image_size), resample=PIL_BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()


def build_weight_map(difference: torch.Tensor, highlight_weight: float, quantile: float) -> torch.Tensor:
    highlight_strength = difference.abs().mean(dim=0, keepdim=True)
    scale = torch.quantile(highlight_strength.flatten(), quantile).clamp_min(1e-6)
    focus = (highlight_strength / scale).clamp(0.0, 1.0)
    return 1.0 + highlight_weight * focus


def rgb_to_luminance(image: torch.Tensor) -> torch.Tensor:
    weights = image.new_tensor([0.2990, 0.5870, 0.1140]).view(3, 1, 1)
    return (image * weights).sum(dim=0, keepdim=True)


def conv2d_single_channel(image: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    return F.conv2d(image.unsqueeze(0), kernel.to(image.device, image.dtype), padding=1).squeeze(0)


def make_edge_condition(glossy: torch.Tensor) -> torch.Tensor:
    gray = rgb_to_luminance(glossy)
    clip_value = torch.quantile(gray.flatten(), 0.98).clamp_min(1e-6)
    gray = gray.clamp(max=clip_value) / clip_value

    blur_kernel = torch.tensor(
        [[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3) / 16.0
    gray = conv2d_single_channel(gray, blur_kernel)

    sobel_x = torch.tensor(
        [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3)
    sobel_y = torch.tensor(
        [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3)

    grad_x = conv2d_single_channel(gray, sobel_x)
    grad_y = conv2d_single_channel(gray, sobel_y)
    edges = torch.sqrt(grad_x.pow(2) + grad_y.pow(2) + 1e-12)
    return edges / edges.amax().clamp_min(1e-6)


def make_condition(glossy: torch.Tensor, condition_mode: str) -> torch.Tensor:
    if condition_mode == 'glossy_rgb':
        return glossy
    return make_edge_condition(glossy)


class PairedPSDDiffuseDataset(Dataset):
    def __init__(self, glossy_dir: Path, diffuse_dir: Path, image_size: int, condition_mode: str, highlight_weight: float, highlight_quantile: float):
        self.glossy_dir = Path(glossy_dir)
        self.diffuse_dir = Path(diffuse_dir)
        self.image_size = int(image_size)
        self.condition_mode = condition_mode
        self.highlight_weight = float(highlight_weight)
        self.highlight_quantile = float(highlight_quantile)

        if not self.glossy_dir.exists():
            raise FileNotFoundError(f'Missing glossy directory: {self.glossy_dir}')
        if not self.diffuse_dir.exists():
            raise FileNotFoundError(f'Missing diffuse directory: {self.diffuse_dir}')

        self.samples = pair_image_paths(self.glossy_dir, self.diffuse_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        glossy_path, diffuse_path, name = self.samples[idx]
        glossy = load_rgb_tensor(glossy_path, self.image_size)
        diffuse = load_rgb_tensor(diffuse_path, self.image_size)
        difference = glossy - diffuse
        condition = make_condition(glossy, self.condition_mode)
        weight_map = build_weight_map(difference, self.highlight_weight, self.highlight_quantile)
        meta = {
            'name': name,
            'glossy_path': str(glossy_path),
            'diffuse_path': str(diffuse_path),
        }
        return glossy, condition, diffuse, weight_map, meta


train_dataset = PairedPSDDiffuseDataset(
    TRAIN_GLOSSY_DIR,
    TRAIN_DIFFUSE_DIR,
    image_size=IMAGE_SIZE,
    condition_mode=CONDITION_MODE,
    highlight_weight=HIGHLIGHT_WEIGHT,
    highlight_quantile=HIGHLIGHT_QUANTILE,
)
val_dataset = PairedPSDDiffuseDataset(
    VAL_GLOSSY_DIR,
    VAL_DIFFUSE_DIR,
    image_size=IMAGE_SIZE,
    condition_mode=CONDITION_MODE,
    highlight_weight=HIGHLIGHT_WEIGHT,
    highlight_quantile=HIGHLIGHT_QUANTILE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

fixed_val_glossy, fixed_val_condition, fixed_val_diffuse, fixed_val_weight_map, fixed_val_meta = val_dataset[VAL_CASE_INDEX]
fixed_val_glossy = fixed_val_glossy.unsqueeze(0).to(DEVICE)
fixed_val_condition = fixed_val_condition.unsqueeze(0).to(DEVICE)
fixed_val_diffuse = fixed_val_diffuse.unsqueeze(0).to(DEVICE)

fixed_eval_generator = torch.Generator()
fixed_eval_generator.manual_seed(EVAL_SEED)
fixed_eval_noise = torch.randn(fixed_val_diffuse.shape, generator=fixed_eval_generator, dtype=fixed_val_diffuse.dtype).to(DEVICE)

print(f'train dataset size: {len(train_dataset)}')
print(f'val dataset size: {len(val_dataset)}')
print(f'train batches per epoch: {len(train_loader)}')
print(f'fixed val case: {fixed_val_meta["name"]}')
print(f'fixed glossy shape: {tuple(fixed_val_glossy.shape)}')
print(f'fixed condition shape: {tuple(fixed_val_condition.shape)}')
print(f'fixed diffuse shape: {tuple(fixed_val_diffuse.shape)}')


In [ ]:
def get_cosine_lambda(initial_lr: float, final_lr: float, epochs: int, warmup_epoch: int):
    def cosine_lambda(epoch_idx: int) -> float:
        if epoch_idx < warmup_epoch:
            return epoch_idx / max(warmup_epoch, 1)
        cosine = (math.cos((epoch_idx - warmup_epoch) / max(epochs - warmup_epoch, 1) * math.pi) + 1.0) / 2.0
        return 1.0 - (1.0 - cosine) * (1.0 - final_lr / initial_lr)

    return cosine_lambda


def checkpoint_save(model, optimizer, loss: float, epoch: int, save_path: Path, condition_mode: str):
    model_dir = Path(f'{save_path}_epoch_{epoch}_loss_{loss:.4f}')
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': int(epoch),
            'loss': float(loss),
            'train_type': TRAIN_TYPE,
            'condition_mode': condition_mode,
            'condition_channels': CONDITION_CHANNELS,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },
        model_dir / 'model.pth',
    )


def update_ema(ema_model, model, decay: float = 0.9999):
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())
    for name, param in model_params.items():
        if name in ema_params:
            ema_params[name].data.mul_(decay).add_(param.data, alpha=1.0 - decay)


def diffuse_loss(prediction: torch.Tensor, target: torch.Tensor, weight_map: torch.Tensor, loss_kind: str) -> tuple[torch.Tensor, torch.Tensor]:
    if loss_kind == 'l1':
        global_loss = F.l1_loss(prediction, target)
        weighted_loss = (weight_map * (prediction - target).abs()).mean()
    else:
        global_loss = F.mse_loss(prediction, target)
        weighted_loss = (weight_map * (prediction - target).pow(2)).mean()
    return global_loss, weighted_loss


def build_restoration_prior(glossy: torch.Tensor) -> torch.Tensor:
    return glossy.clamp(0.0, 1.0)


def target_space_limits() -> tuple[float, float]:
    return 0.0, 1.0


class ConditionedSDEUNet(SDEBackbone.UNetWithTransformer):
    def __init__(self, noise_steps: int = 1000, time_dim: int = 256, size: int = 32, depth: int = 4, conditioning_channels: int = 1):
        super().__init__(noise_steps=noise_steps, time_dim=time_dim, size=size, depth=depth)
        self.conditioning_channels = conditioning_channels
        self.dit = Transformer.Transformer_S_2(
            input_size=self.image_size // (2 ** self.depth),
            in_channels=self.dit_channels,
            conditioning_channels=conditioning_channels,
            learn_sigma=False,
        )
        self.dit_proj_in = nn.Conv2d(self.dit_channels, self.dit.in_channels, kernel_size=1)
        self.dit_proj_out = nn.Conv2d(self.dit.out_channels, self.dit_channels, kernel_size=1)


def restore_sample_x(
    model: torch.nn.Module,
    diffuser: diff.Diffuser,
    glossy: torch.Tensor,
    condition: torch.Tensor,
    start_step: int,
    deterministic: bool,
    initial_noise: torch.Tensor | None = None,
    clip_x0: bool = True,
) -> torch.Tensor:
    if not 0 <= start_step < diffuser.steps:
        raise ValueError(f'start_step must be in [0, {diffuser.steps - 1}], got {start_step}')

    prior = build_restoration_prior(glossy)
    if initial_noise is None:
        noise = torch.randn_like(prior)
    else:
        noise = initial_noise.to(glossy.device, dtype=glossy.dtype)

    t0 = torch.full((glossy.shape[0],), start_step, dtype=torch.long, device=glossy.device)
    x_t = diffuser.forward_diffusion(prior, t0, noise)

    with torch.no_grad():
        for step in range(start_step, -1, -1):
            t = torch.full((glossy.shape[0],), step, dtype=torch.long, device=glossy.device)
            model_output = model(x_t, t, condition)
            pred_x0, _ = diffuser.predict_x0_and_noise(x_t, t, model_output, parameterization='x')

            if clip_x0:
                low, high = target_space_limits()
                pred_x0 = pred_x0.clamp(low, high)

            if step == 0:
                return pred_x0

            mean, variance = diffuser.q_posterior(pred_x0, x_t, t)
            if deterministic:
                x_t = mean
            else:
                x_t = mean + torch.sqrt(variance.clamp_min(1e-20)) * torch.randn_like(x_t)

    return x_t


model = ConditionedSDEUNet(
    noise_steps=NOISE_STEPS,
    size=IMAGE_SIZE,
    depth=DEPTH,
    conditioning_channels=CONDITION_CHANNELS,
).to(DEVICE)
ema_model = deepcopy(model).to(DEVICE)
ema_model.load_state_dict(model.state_dict())
ema_model.eval()

diffuser = diff.CosSchDiffuser(steps=NOISE_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_cosine_lambda(initial_lr=LR, final_lr=FINAL_LR, epochs=EPOCHS, warmup_epoch=WARMUP_EPOCHS),
)

num_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'model: {model.__class__.__name__}')
print(f'trainable parameters: {num_params:,}')
print(f'optimizer: AdamW(lr={LR})')
print(f'diffuser: {diffuser.name}, steps={diffuser.steps}')


def compute_step_outputs(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device, loss_kind: str, global_weight: float, weighted_weight: float):
    glossy, condition, diffuse, weight_map, _meta = batch
    glossy = glossy.to(device)
    condition = condition.to(device)
    diffuse = diffuse.to(device)
    weight_map = weight_map.to(device)

    batch_size = diffuse.shape[0]
    t = torch.randint(0, diffuser.steps, (batch_size,), dtype=torch.long, device=device)
    noise = torch.randn_like(diffuse)
    noisy_xt = diffuser.forward_diffusion(diffuse, t, noise)
    prediction = model(noisy_xt, t, condition)

    global_loss, weighted_loss = diffuse_loss(prediction, diffuse, weight_map, loss_kind)
    total_loss = global_weight * global_loss + weighted_weight * weighted_loss

    return {
        'total_loss': total_loss,
        'global_loss': global_loss.detach(),
        'weighted_loss': weighted_loss.detach(),
        'prediction': prediction,
        'diffuse': diffuse,
        'glossy': glossy,
        'condition': condition,
    }


def evaluate_objective(model: torch.nn.Module, loader, diffuser: diff.Diffuser, device: torch.device, loss_kind: str, global_weight: float, weighted_weight: float, max_batches: int | None = None) -> dict:
    was_training = model.training
    model.eval()
    totals, globals_, weighteds = [], [], []
    with torch.no_grad():
        for idx, batch in enumerate(loader):
            if max_batches is not None and idx >= max_batches:
                break
            outputs = compute_step_outputs(model, batch, diffuser, device, loss_kind, global_weight, weighted_weight)
            totals.append(float(outputs['total_loss'].item()))
            globals_.append(float(outputs['global_loss'].item()))
            weighteds.append(float(outputs['weighted_loss'].item()))
    if was_training:
        model.train()
    return {
        'objective': float(np.mean(totals)) if totals else float('nan'),
        'global_loss': float(np.mean(globals_)) if globals_ else float('nan'),
        'weighted_loss': float(np.mean(weighteds)) if weighteds else float('nan'),
    }


def channel_limits(a: torch.Tensor, b: torch.Tensor):
    low = min(float(a.min()), float(b.min()))
    high = max(float(a.max()), float(b.max()))
    if abs(high - low) < 1e-8:
        high = low + 1e-8
    return low, high


def plot_condition(ax, condition: torch.Tensor, condition_mode: str):
    if condition_mode == 'glossy_rgb':
        ax.imshow(condition.permute(1, 2, 0).numpy().clip(0.0, 1.0))
        ax.set_title('Condition glossy RGB')
    else:
        ax.imshow(condition.squeeze(0).numpy(), cmap='gray')
        ax.set_title('Condition edge map')
    ax.axis('off')


def plot_prediction(glossy_tensor: torch.Tensor, condition_tensor: torch.Tensor, target_tensor: torch.Tensor, prediction_tensor: torch.Tensor, epoch: int, condition_mode: str):
    glossy = glossy_tensor.detach().cpu().squeeze(0)
    condition = condition_tensor.detach().cpu().squeeze(0)
    target = target_tensor.detach().cpu().squeeze(0)
    prediction = prediction_tensor.detach().cpu().squeeze(0).clamp(0.0, 1.0)

    target_error = (prediction - target).abs()
    diffuse_error = target_error.mean(dim=0)

    fig, axes = plt.subplots(5, 3, figsize=(12, 18))
    for channel_idx, channel_name in enumerate(['Red', 'Green', 'Blue']):
        vmin, vmax = channel_limits(target[channel_idx], prediction[channel_idx])
        axes[channel_idx, 0].imshow(target[channel_idx], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[channel_idx, 0].set_title(f'{channel_name} diffuse target')
        axes[channel_idx, 1].imshow(prediction[channel_idx], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[channel_idx, 1].set_title(f'{channel_name} diffuse prediction')
        axes[channel_idx, 2].imshow(target_error[channel_idx], cmap='magma')
        axes[channel_idx, 2].set_title(f'{channel_name} abs error')

    axes[3, 0].imshow(target.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[3, 0].set_title('RGB diffuse target')
    axes[3, 1].imshow(prediction.permute(1, 2, 0).numpy())
    axes[3, 1].set_title('RGB diffuse prediction')
    axes[3, 2].imshow(diffuse_error.numpy(), cmap='magma')
    axes[3, 2].set_title('RGB mean abs error')

    axes[4, 0].imshow(glossy.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 0].set_title('Glossy input')
    plot_condition(axes[4, 1], condition, condition_mode)
    axes[4, 2].imshow(diffuse_error.numpy(), cmap='magma')
    axes[4, 2].set_title('Diffuse abs error')

    for ax in axes.ravel():
        if ax is not axes[4, 1]:
            ax.axis('off')

    fig.suptitle(f'Fixed validation restoration at epoch {epoch} | {condition_mode}', fontsize=16)
    plt.tight_layout()
    plt.show()


def run_eval(model: torch.nn.Module, loader, diffuser: diff.Diffuser, fixed_glossy: torch.Tensor, fixed_condition: torch.Tensor, fixed_diffuse: torch.Tensor, fixed_noise: torch.Tensor, device: torch.device, epoch: int, loss_kind: str, global_weight: float, weighted_weight: float, max_batches: int | None = None):
    objective_metrics = evaluate_objective(model, loader, diffuser, device, loss_kind, global_weight, weighted_weight, max_batches=max_batches)
    with torch.no_grad():
        prediction = restore_sample_x(
            model,
            diffuser,
            fixed_glossy,
            fixed_condition,
            start_step=RESTORE_START_STEP,
            deterministic=RESTORE_DETERMINISTIC,
            initial_noise=fixed_noise,
            clip_x0=RESTORE_CLIP_X0,
        )

    target = fixed_diffuse.squeeze(0)
    predicted = prediction.squeeze(0)
    diffuse_mse = float(F.mse_loss(predicted, target).item())
    diffuse_mae = float((predicted - target).abs().mean().item())
    diffuse_psnr = float((10.0 * torch.log10(1.0 / F.mse_loss(predicted.clamp(0.0, 1.0), target).clamp_min(1e-10))).item())

    plot_prediction(fixed_glossy, fixed_condition, fixed_diffuse, prediction, epoch=epoch, condition_mode=CONDITION_MODE)
    return {
        'epoch': epoch,
        'val_objective': objective_metrics['objective'],
        'val_global_loss': objective_metrics['global_loss'],
        'val_weighted_loss': objective_metrics['weighted_loss'],
        'diffuse_mse': diffuse_mse,
        'diffuse_mae': diffuse_mae,
        'diffuse_psnr': diffuse_psnr,
        'restore_start_step': RESTORE_START_STEP,
    }


In [ ]:
progress_bar = tqdm(total=EPOCHS * len(train_loader), desc=f'Training [SDE | {CONDITION_MODE}]', dynamic_ncols=True)
train_total_history = []
train_global_history = []
train_weighted_history = []
eval_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_total = 0.0
    epoch_global = 0.0
    epoch_weighted = 0.0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        outputs = compute_step_outputs(model, batch, diffuser, DEVICE, LOSS_KIND, GLOBAL_LOSS_WEIGHT, WEIGHTED_LOSS_WEIGHT)
        loss = outputs['total_loss']
        loss.backward()
        optimizer.step()
        update_ema(ema_model, model, decay=EMA_DECAY)

        epoch_total += float(outputs['total_loss'].item())
        epoch_global += float(outputs['global_loss'].item())
        epoch_weighted += float(outputs['weighted_loss'].item())
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    epoch_total /= len(train_loader)
    epoch_global /= len(train_loader)
    epoch_weighted /= len(train_loader)
    train_total_history.append(epoch_total)
    train_global_history.append(epoch_global)
    train_weighted_history.append(epoch_weighted)
    scheduler.step()

    print(f'Epoch {epoch:4d} | total {epoch_total:.6f} | global {epoch_global:.6f} | weighted {epoch_weighted:.6f}')

    if epoch % CHECKPOINT_EVERY == 0:
        checkpoint_save(model, optimizer, epoch_total, epoch, TRAINER_SAVE_PATH, CONDITION_MODE)

    if epoch % EVAL_EVERY == 0:
        metrics = run_eval(
            ema_model,
            val_loader,
            diffuser,
            fixed_val_glossy,
            fixed_val_condition,
            fixed_val_diffuse,
            fixed_eval_noise,
            DEVICE,
            epoch,
            loss_kind=LOSS_KIND,
            global_weight=GLOBAL_LOSS_WEIGHT,
            weighted_weight=WEIGHTED_LOSS_WEIGHT,
            max_batches=VAL_BATCHES,
        )
        eval_history.append(metrics)
        print(
            f"Eval {epoch:4d} | restore@{RESTORE_START_STEP} | val objective {metrics['val_objective']:.6f} | "
            f"diffuse mse {metrics['diffuse_mse']:.6f} | diffuse psnr {metrics['diffuse_psnr']:.4f} dB"
        )

progress_bar.close()

torch.save(
    {
        'epoch': EPOCHS,
        'train_type': TRAIN_TYPE,
        'condition_mode': CONDITION_MODE,
        'condition_channels': CONDITION_CHANNELS,
        'restore_start_step': RESTORE_START_STEP,
        'model_state_dict': model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_total_history': train_total_history,
        'train_global_history': train_global_history,
        'train_weighted_history': train_weighted_history,
        'eval_history': eval_history,
    },
    FINAL_MODEL_PATH,
)
print(f'Final model saved to {FINAL_MODEL_PATH}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(train_total_history) + 1), train_total_history, label='total loss')
axes[0].plot(range(1, len(train_global_history) + 1), train_global_history, label='global diffuse loss')
axes[0].plot(range(1, len(train_weighted_history) + 1), train_weighted_history, label='weighted highlight loss')
axes[0].set_title(f'Train Losses ({CONDITION_MODE})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)
axes[0].legend()

if eval_history:
    eval_epochs = [item['epoch'] for item in eval_history]
    axes[1].plot(eval_epochs, [item['val_objective'] for item in eval_history], label='val objective')
    axes[1].plot(eval_epochs, [item['diffuse_mse'] for item in eval_history], label='diffuse mse')
    axes[1].plot(eval_epochs, [item['diffuse_psnr'] for item in eval_history], label='diffuse psnr')
    axes[1].set_title(f'Restoration Validation Curves ({CONDITION_MODE})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric')
    axes[1].grid(True)
    axes[1].legend()
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()
